In [63]:
import numpy as np

np.random.seed(42)

Importar os dados

In [64]:
#importar dados brutos

import pandas as pd

data2023a1 = pd.read_csv(
    "dados_brutos/area1_2023.csv",
    encoding="latin-1",
    sep=";",
)

data2023a2 = pd.read_csv(
    "dados_brutos/area2_2023.csv",
    encoding="latin-1",
    sep=";",
)

data2024a1 = pd.read_csv(
    "dados_brutos/area1_2024.csv",
    encoding="latin-1",
    sep=";",
)

data2024a2 = pd.read_csv(
    "dados_brutos/area2_2024.csv",
    encoding="latin-1",
    sep=";",
)

data2025a1 = pd.read_csv(
    "dados_brutos/area1_2025_completo.csv",
    encoding="latin-1",
    sep=";",
)

data2025a2 = pd.read_csv(
    "dados_brutos/area2_2025_completo.csv",
    encoding="latin-1",
    sep=";",
)

data2023temp = pd.read_csv(
    "dados_brutos/INMET_SE_SP_A711_SAO CARLOS_01-01-2023_A_31-12-2023.CSV",
    encoding="latin-1",
    sep=";",
    skiprows=8
)

data2024temp = pd.read_csv(
    "dados_brutos/INMET_SE_SP_A711_SAO CARLOS_01-01-2024_A_31-12-2024.CSV",
    encoding="latin-1",
    sep=";",
    skiprows=8
)

data2025temp = pd.read_csv(
    "dados_brutos/INMET_SE_SP_A711_SAO CARLOS_01-01-2025_A_31-12-2025.CSV",
    encoding="latin-1",
    sep=";",
    skiprows=8
)

datacardapio = pd.read_csv('dados_brutos/cardapio.txt', delimiter=',')
datacardapio.to_csv('cardapio.csv', index=False)

agrupamento = pd.ExcelFile('dados_brutos/agrupamento_cardapio.xlsx')

Tratamento dos arquivos de demanda

In [65]:
#Padronizar a grafia dos dias da semana

data2023a1['Dia_semana'] = data2023a1['Dia_semana'].str.lower()
data2024a1['Dia_semana'] = data2024a1['Dia_semana'].str.lower()
data2025a1['Dia_semana'] = data2025a1['Dia_semana'].str.lower()
data2023a2['Dia_semana'] = data2023a2['Dia_semana'].str.lower()
data2024a2['Dia_semana'] = data2024a2['Dia_semana'].str.lower()
data2025a2['Dia_semana'] = data2025a2['Dia_semana'].str.lower()

In [66]:
def transformar(df, ano, area):

    if area == 1:
        df = df[['Mês', 'Dia_semana', 'Dia_mês', 'Total_almoço', 'Total_jantar']]

        df_long = df.melt(
            id_vars=['Mês', 'Dia_semana', 'Dia_mês'],
            value_vars=['Total_almoço', 'Total_jantar'],
            var_name='refeicao',
            value_name='total'
        )

        df_long['refeicao'] = df_long['refeicao'].replace({
            'Total_almoço': 'almoco',
            'Total_jantar': 'jantar'
        })

        ordem = ['almoco', 'jantar']
        df_long['refeicao'] = pd.Categorical(
            df_long['refeicao'],
            categories=ordem,
            ordered=True
        )

    else:  # área 2

        df = df[['Mês', 'Dia_semana', 'Dia_mês', 'Total_almoço']]
        df_long = df.rename(columns={'Total_almoço': 'total'})
        df_long['refeicao'] = 'almoco'

    # adiciona coluna ano
    df_long.insert(0, 'Ano', ano)

    # ordenação
    df_long = df_long.sort_values(
        by=['Ano', 'Mês', 'Dia_mês', 'refeicao', 'total']
    ).reset_index(drop=True)

    df_long = df_long[['Ano', 'Mês', 'Dia_semana', 'Dia_mês', 'refeicao', 'total']]
    return df_long

In [67]:
data2023a1 = transformar(data2023a1,2023,1)
data2024a1 = transformar(data2024a1,2024,1)
data2025a1 = transformar(data2025a1,2025,1)

data2023a2 = transformar(data2023a2,2023,2)
data2024a2= transformar(data2024a2,2024,2)
data2025a2= transformar(data2025a2,2025,2)

Tratamento dos arquivos de informações meteorológicas

In [68]:
def processa_temp(df):

    # Remove colunas que não serão usadas
    df = df.drop(columns=[
        'PRESSAO ATMOSFERICA AO NIVEL DA ESTACAO, HORARIA (mB)',
        'PRESSÃO ATMOSFERICA MAX.NA HORA ANT. (AUT) (mB)',
        'PRESSÃO ATMOSFERICA MIN. NA HORA ANT. (AUT) (mB)',
        'RADIACAO GLOBAL (Kj/m²)',
        'TEMPERATURA DO AR - BULBO SECO, HORARIA (°C)',
        'TEMPERATURA DO PONTO DE ORVALHO (°C)',
        'TEMPERATURA ORVALHO MAX. NA HORA ANT. (AUT) (°C)',
        'TEMPERATURA ORVALHO MIN. NA HORA ANT. (AUT) (°C)',
        'UMIDADE REL. MAX. NA HORA ANT. (AUT) (%)',
        'UMIDADE REL. MIN. NA HORA ANT. (AUT) (%)',
        'VENTO, DIREÇÃO HORARIA (gr) (° (gr))',
        'Unnamed: 19'
    ], errors='ignore')

    # Cria datetime
    df['datetime'] = pd.to_datetime(
        df['Data'] + ' ' + df['Hora UTC'],
        format='%Y/%m/%d %H%M UTC',
        errors='coerce'
    )

    # Extrai data e hora
    df['data'] = df['datetime'].dt.date
    df['hora'] = df['datetime'].dt.hour

    # Colunas numéricas
    colunas_numericas = [
        'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)',
        'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)',
        'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)',
        'UMIDADE RELATIVA DO AR, HORARIA (%)',
        'VENTO, VELOCIDADE HORARIA (m/s)',
        'VENTO, RAJADA MAXIMA (m/s)'
    ]

    # Converte para numérico
    for col in colunas_numericas:
        df[col] = pd.to_numeric(
            df[col].astype(str).str.replace(',', '.', regex=False),
            errors='coerce'
        )

    # Renomeia colunas
    df = df.rename(columns={
        'PRECIPITAÇÃO TOTAL, HORÁRIO (mm)': 'Precipitacao_mm',
        'TEMPERATURA MÁXIMA NA HORA ANT. (AUT) (°C)': 'Temp_max_C',
        'TEMPERATURA MÍNIMA NA HORA ANT. (AUT) (°C)': 'Temp_min_C',
        'UMIDADE RELATIVA DO AR, HORARIA (%)': 'Umid_rel_ar',
        'VENTO, RAJADA MAXIMA (m/s)': 'Vento_rajada_maxima_ms',
        'VENTO, VELOCIDADE HORARIA (m/s)': 'Vento_velocidade_ms'
    })

    # Atualiza lista de colunas
    colunas_numericas = [
        'Precipitacao_mm',
        'Temp_max_C',
        'Temp_min_C',
        'Umid_rel_ar',
        'Vento_velocidade_ms',
        'Vento_rajada_maxima_ms'
    ]

    # Classifica refeição
    def classifica_refeicao(h):
        if 10 <= h <= 11:
            return 'cafe da manha'
        elif 14 <= h <= 16:
            return 'almoco'
        elif 20 <= h <= 22:
            return 'jantar'
        else:
            return None

    df['refeicao'] = df['hora'].apply(classifica_refeicao)

    # Mantém só refeições desejadas
    df = df[df['refeicao'].notna()]

    # Agrupa
    df_refeicao = (
        df.groupby(['data', 'refeicao'])[colunas_numericas]
        .mean()
        .reset_index()
    )

    # Data → Ano/Mês/Dia
    df_refeicao['data'] = pd.to_datetime(df_refeicao['data'])
    df_refeicao['Ano'] = df_refeicao['data'].dt.year
    df_refeicao['Mês'] = df_refeicao['data'].dt.month
    df_refeicao['Dia_mês'] = df_refeicao['data'].dt.day

    # Reorganiza colunas
    df_refeicao = df_refeicao[['Ano','Mês','Dia_mês','refeicao'] + colunas_numericas]

    ordem_refeicao = ['cafe da manha','almoco','jantar']
    df_refeicao['refeicao'] = pd.Categorical(df_refeicao['refeicao'], categories=ordem_refeicao, ordered=True)

    df_refeicao = df_refeicao.sort_values(by=['Ano','Mês','Dia_mês','refeicao']).reset_index(drop=True)

    return df_refeicao

In [69]:
#Aplica a função

data2023_meteorologia = processa_temp(data2023temp)
data2024_meteorologia = processa_temp(data2024temp)
data2025_meteorologia = processa_temp(data2025temp)

Tratamento dos dados de calendário

In [70]:
import pandas as pd

#Criar calendário
df = pd.DataFrame({
    'data': pd.date_range(start='2023-01-01', end='2025-12-31')
})

#Extrair informações de data
df['Ano'] = df['data'].dt.year
df['Mês'] = df['data'].dt.month
df['Dia_mês'] = df['data'].dt.day
df['Dia_semana'] = df['data'].dt.weekday  # 0=segunda, ..., 6=domingo

#Segunda (0) a sábado (5) = letivo, domingo (6) = não letivo
df['letivo'] = df['Dia_semana'] != 6 

#Definir feriados
feriados = [
    ('2023-04-03', '2023-04-08'), #semana santa
    ('2023-04-21', '2023-04-22'), #tiradentes
    '2023-05-01', #dia do trabalho
    ('2023-06-08', '2023-06-10'), #corpus christi
    ('2023-08-14','2023-08-15'), #feriado municipal
    ('2023-09-04', '2023-09-09'), #semana da pátria
    ('2023-10-12', '2023-10-14'), #dia da padroeira do Brasil
    '2023-10-28', #consagração ao funcionário público
    ('2023-11-02', '2023-11-04'), #finados
    '2023-11-15', #proclamação da república

    ('2024-03-25', '2024-03-30'), #semana santa
    '2024-05-01', #dia do trabalho
    ('2024-05-30', '2024-06-01'), #corpus christi
    ('2024-08-15','2024-08-17'), #feriado municipal
    ('2024-09-02', '2024-09-07'), #semana da pátria
    '2024-10-12', #dia da padroeira do Brasil
    '2024-10-28', #consagração ao funcionário público
    '2024-11-02', #finados
    '2024-11-04', #feriado municipal
    ('2024-11-15', '2024-11-16'), #proclamação da república
    '2024-11-20', #dia da consciência negra

    ('2025-03-03', '2025-03-05'), #carnaval + quarta feira de cinzas
    ('2025-04-14', '2025-04-19'), #semana santa
    '2025-04-21', #tiradentes
    ('2025-05-01', '2025-05-03'), #dia do trabalho
    ('2025-06-19', '2025-06-21'), #corpus christi
    ('2025-08-15', '2025-08-16'), #feriado municipal
    ('2025-09-01', '2025-09-06'), #semana da pátria
    ('2025-10-27', '2025-10-28'), #consagração ao funcionário público
    ('2025-11-03', '2025-11-04'), #feriado municipal
    '2025-11-15', #proclamação da república
    ('2025-11-20', '2025-11-22') #dia da consciência negra
]

df['feriado'] = False

for f in feriados:
    if isinstance(f, tuple):  # intervalo
        inicio, fim = f
        df.loc[df['data'].between(inicio, fim), 'feriado'] = True
    else:  # data única
        df.loc[df['data'] == f, 'feriado'] = True

#Definir férias (intervalos)
ferias = [
    ('2023-01-01', '2023-03-12'),
    ('2023-07-15', '2023-08-06'),

    ('2023-12-21', '2024-02-25'),
    ('2024-07-02', '2024-08-04'),

    ('2024-12-12', '2025-02-23'),
    ('2025-07-07', '2025-08-03'),
    ('2025-12-12','2025-12-31')
]

df['ferias'] = False
for inicio, fim in ferias:
    mask = (df['data'] >= inicio) & (df['data'] <= fim)
    df.loc[mask, 'ferias'] = True

# feriado ou férias NÃO são letivos
df.loc[df['feriado'] | df['ferias'], 'letivo'] = False

df['Dia_semana'] = df['Dia_semana'].map({
    0: 'seg',
    1: 'ter',
    2: 'qua',
    3: 'qui',
    4: 'sex',
    5: 'sáb',
    6: 'dom'
})

df['letivo'] = df['letivo'].astype(int)
df['feriado'] = df['feriado'].astype(int)
df['ferias'] = df['ferias'].astype(int)

data_calendario = df.drop(columns=['data'])

In [71]:
import pandas as pd
import numpy as np

# 1. Preparação: Criação da coluna de data_calendario real e ordenação
data_calendario['_data'] = pd.to_datetime(
    data_calendario['Ano'].astype(str) + '-' +
    data_calendario['Mês'].astype(str).str.zfill(2) + '-' +
    data_calendario['Dia_mês'].astype(str).str.zfill(2)
)
data_calendario = data_calendario.sort_values('_data').reset_index(drop=True)

# 2. Vésperas e Pós (Janelas de 1 dia)
# Véspera: Olha para o dia seguinte (shift -1) | Pós: Olha para o dia anterior (shift 1)
data_calendario['vespera_nao_letivo'] = (data_calendario['letivo'].shift(-1) == 0).astype(int)
data_calendario['pos_nao_letivo'] = (data_calendario['letivo'].shift(1) == 0).astype(int)

data_calendario['vespera_feriado'] = (data_calendario['feriado'].shift(-1) == 1).astype(int)
data_calendario['pos_feriado'] = (data_calendario['feriado'].shift(1) == 1).astype(int)

# 3. Lógica de Férias (Usando fill_value=0 na primeira linha)
# Máscaras para encontrar os dias exatos de transição
inicio_ferias_mask = (data_calendario['ferias'] == 1) & (data_calendario['ferias'].shift(1, fill_value=0) == 0)
retorno_aulas_mask = (data_calendario['ferias'] == 0) & (data_calendario['ferias'].shift(1) == 1)

# DIAS DESDE O INÍCIO DAS FÉRIAS ATUAIS
# Pega a data_calendario de início e arrasta para o futuro (ffill)
data_calendario['_dt_start_current'] = data_calendario.where(inicio_ferias_mask)['_data'].ffill()
data_calendario['dias_desde_inicio_ferias'] = (data_calendario['_data'] - data_calendario['_dt_start_current']).dt.days.fillna(0).astype(int)
data_calendario.loc[data_calendario['ferias'] == 0, 'dias_desde_inicio_ferias'] = 0

# DIAS DESDE AS ÚLTIMAS FÉRIAS (Foco em dias de aula)
# Pega a data_calendario de retorno às aulas e arrasta para o futuro (ffill)
data_calendario['_dt_last_end'] = data_calendario.where(retorno_aulas_mask)['_data'].ffill()
data_calendario['dias_desde_ultimas_ferias'] = (data_calendario['_data'] - data_calendario['_dt_last_end']).dt.days.fillna(0).astype(int)
data_calendario.loc[data_calendario['ferias'] == 1, 'dias_desde_ultimas_ferias'] = 0

# DIAS ATÉ AS PRÓXIMAS FÉRIAS (Contagem Regressiva)
# Pega a data_calendario do próximo início e arrasta para o passado (bfill)
data_calendario['_dt_next_start'] = data_calendario.where(inicio_ferias_mask)['_data'].bfill()
data_calendario['dias_ate_ferias'] = (data_calendario['_dt_next_start'] - data_calendario['_data']).dt.days.fillna(0).astype(int)
data_calendario.loc[data_calendario['ferias'] == 1, 'dias_ate_ferias'] = 0


# 4. Lógica de Feriados (Contagem Regressiva)
# Encontra o 1º dia de cada bloco de feriado
inicio_feriado_mask = (data_calendario['feriado'] == 1) & (data_calendario['feriado'].shift(1, fill_value=0) == 0)

# Pega essa data_calendario do feriado e arrasta para o passado (bfill)
data_calendario['_dt_next_feriado'] = data_calendario.where(inicio_feriado_mask)['_data'].bfill()
data_calendario['dias_ate_feriado'] = (data_calendario['_dt_next_feriado'] - data_calendario['_data']).dt.days.fillna(0).astype(int)
data_calendario.loc[data_calendario['feriado'] == 1, 'dias_ate_feriado'] = 0


# 5. Limpeza de Memória (Removendo colunas auxiliares)
data_calendario = data_calendario.drop(columns=[
    '_data', 
    '_dt_start_current', 
    '_dt_last_end', 
    '_dt_next_start', 
    '_dt_next_feriado'
])


Tratamentos de dados do cardápio

In [72]:
import pandas as pd

datacardapio[['sobremesa_1', 'sobremesa_2']] = datacardapio['sobremesa'].str.split(' e ', n=1, expand=True)

datacardapio = datacardapio.drop(columns=['sobremesa'])
datacardapio['data'] = datacardapio['data'].str.strip()

datacardapio['data'] = pd.to_datetime(datacardapio['data'], errors='coerce')  # erros viram NaT

datacardapio['Ano'] = datacardapio['data'].dt.year
datacardapio['Mês'] = datacardapio['data'].dt.month
datacardapio['Dia_mês'] = datacardapio['data'].dt.day

datacardapio = datacardapio.drop(columns=['data'])

datacardapio = datacardapio[['Ano', 'Mês', 'Dia_mês', 'refeicao', 'prato_principal_1',
                             'prato_principal_2', 'guarnição', 'sobremesa_1', 'sobremesa_2']]

datacardapio = datacardapio.drop(columns=['sobremesa_2'])

Unindo dfs

In [73]:
#Juntar anos da demanda

# Concatenando todos
data_demandaa1= pd.concat([data2023a1, data2024a1, data2025a1], ignore_index=True, sort=False)
data_demandaa2= pd.concat([data2023a2, data2024a2, data2025a2], ignore_index=True, sort=False)

# Reorganizando para colocar 'Ano' como primeira coluna
colunas = ['Ano'] + [c for c in data_demandaa1.columns if c != 'Ano']
dataarea1 = data_demandaa1[colunas]

In [74]:
#Juntar demanda e cardápio

df1a1 = pd.merge(data_demandaa1, datacardapio, on=['Ano', 'Mês', 'Dia_mês', 'refeicao'])
df1a2 = pd.merge(data_demandaa2, datacardapio, on=['Ano', 'Mês', 'Dia_mês', 'refeicao'])

In [75]:
#Juntar anos metereológicos

# Concatenando todos
data_met= pd.concat([data2023_meteorologia, data2024_meteorologia, data2025_meteorologia], ignore_index=True, sort=False)

In [76]:
#Juntar dados metereológicos

df2a1 = pd.merge(df1a1, data_met, on=['Ano', 'Mês', 'Dia_mês', 'refeicao'])
df2a2 = pd.merge(df1a2, data_met, on=['Ano', 'Mês', 'Dia_mês', 'refeicao'])

In [77]:
#Juntar calendario

df_finala1 = pd.merge(df2a1, data_calendario, on=['Ano', 'Mês', 'Dia_mês','Dia_semana'])
df_finala2 = pd.merge(df2a2, data_calendario, on=['Ano', 'Mês', 'Dia_mês','Dia_semana'])

In [78]:
#Agrupar cardápio

def agrupar_nomes(data, xlsx, sheet_name, column_in_data):
  data_agrupamento = pd.read_excel(xlsx, sheet_name, header=(0))

  # Transpor para ficar mais fácil de tratar
  data_agrupamento_transposed = data_agrupamento.T

  #Itere pela coluna, pegue todos os valores dela
  grupos = []

  for name, col in data_agrupamento_transposed.items():
    #Remove os nans
    categoria_variacoes = list(filter(lambda x: not pd.isna(x), col.values))
    grupos.append(categoria_variacoes)


  # -----------------------
  # Substitui nomes

  # Criar dicionário
  substituicoes = {}

  #Aplicando a substituição
  for array_categoria in grupos:
    substituicoes[array_categoria[0]] = array_categoria


  for prato_padrao, variacoes in substituicoes.items():
      data[column_in_data] = data[column_in_data].replace(variacoes, prato_padrao)

  return data

In [79]:
#Agrupar cardápio

df_finala1 = agrupar_nomes(df_finala1, agrupamento, 'Carnes', 'prato_principal_1')
df_finala1 = agrupar_nomes(df_finala1, agrupamento, 'Vegetariano', 'prato_principal_2')
df_finala1 = agrupar_nomes(df_finala1, agrupamento, 'Guarnição', 'guarnição')
df_finala1 = agrupar_nomes(df_finala1, agrupamento, 'Sobremesa doce', 'sobremesa_1')

df_finala2 = agrupar_nomes(df_finala2, agrupamento, 'Carnes', 'prato_principal_1')
df_finala2 = agrupar_nomes(df_finala2, agrupamento, 'Vegetariano', 'prato_principal_2')
df_finala2 = agrupar_nomes(df_finala2, agrupamento, 'Guarnição', 'guarnição')
df_finala2 = agrupar_nomes(df_finala2, agrupamento, 'Sobremesa doce', 'sobremesa_1')

In [80]:
df_finala1 = df_finala1.dropna()
df_finala2 = df_finala2.dropna()
df_finala2 = df_finala2.drop(columns=['refeicao'])

In [81]:
df_finala1.to_csv('../../data/dataarea1.csv', index=False)
df_finala2.to_csv('../../data/dataarea2.csv', index=False)

In [82]:
import os

if os.path.exists('../data_treatment_completo/cardapio.csv'):
    os.remove('../data_treatment_completo/cardapio.csv')
